## Data exploration

Let us start by importing our training csv data into R.
We also set a seed so that the results are repeatable.

In [10]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)

set.seed(2) 

In [2]:
dim(data)
dim(data_labels)

[1] 260601     39

[1] 260601      2

In [14]:
head(data,3)
head(data_labels,3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,⋯,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,802906,6,487,12198,2,30,6,5,t,r,⋯,0,0,0,0,0,0,0,0,0,0
2,28830,8,900,2812,2,10,8,7,o,r,⋯,0,0,0,0,0,0,0,0,0,0
3,94947,21,363,8973,2,10,5,5,t,r,⋯,0,0,0,0,0,0,0,0,0,0


,building_id,damage_grade
,<int>,<int>
1,802906,3
2,28830,2
3,94947,3


We merge the training features and targets into the same structure

In [75]:
datam<-merge(data,data_labels,by=c('building_id','building_id'))
head(datam,3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,⋯,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,4,30,266,1224,1,25,5,2,t,r,⋯,0,0,0,0,0,0,0,0,0,2
2,8,17,409,12182,2,0,13,7,t,r,⋯,0,0,0,0,0,0,0,0,0,3
3,12,17,716,7056,2,5,12,6,o,r,⋯,0,0,0,0,0,0,0,0,0,3


Let's check if there are any NA's or empty values.

In [29]:
any(is.na(datam))
any(datam[,]==" ")
any(datam[,]=="")

[1] FALSE

[1] FALSE

[1] FALSE

In [30]:
summary(datam)

  building_id      geo_level_1_id geo_level_2_id   geo_level_3_id 
 Min.   :      4   Min.   : 0.0   Min.   :   0.0   Min.   :    0  
 1st Qu.: 261190   1st Qu.: 7.0   1st Qu.: 350.0   1st Qu.: 3073  
 Median : 525757   Median :12.0   Median : 702.0   Median : 6270  
 Mean   : 525676   Mean   :13.9   Mean   : 701.1   Mean   : 6258  
 3rd Qu.: 789762   3rd Qu.:21.0   3rd Qu.:1050.0   3rd Qu.: 9412  
 Max.   :1052934   Max.   :30.0   Max.   :1427.0   Max.   :12567  
                                                                  
 count_floors_pre_eq      age         area_percentage   height_percentage
 Min.   :1.00        Min.   :  0.00   Min.   :  1.000   Min.   : 2.000   
 1st Qu.:2.00        1st Qu.: 10.00   1st Qu.:  5.000   1st Qu.: 4.000   
 Median :2.00        Median : 15.00   Median :  7.000   Median : 5.000   
 Mean   :2.13        Mean   : 26.54   Mean   :  8.018   Mean   : 5.434   
 3rd Qu.:2.00        3rd Qu.: 30.00   3rd Qu.:  9.000   3rd Qu.: 6.000   
 Max.   :9.00       

Some variables are categorical. We wil now one-hot encode them.

In [76]:
factor_variables<-which(sapply(datam[1,],class)=="factor")
factor_variables

land_surface_condition        foundation_type              roof_type 
                     9                     10                     11 
     ground_floor_type       other_floor_type               position 
                    12                     13                     14 
    plan_configuration legal_ownership_status 
                    15                     27

In [77]:
datam_unfactor<-datam[,-factor_variables]
head(datam_unfactor,3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,⋯,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,4,30,266,1224,1,25,5,2,0,1,⋯,0,0,0,0,0,0,0,0,0,2
2,8,17,409,12182,2,0,13,7,0,1,⋯,0,0,0,0,0,0,0,0,0,3
3,12,17,716,7056,2,5,12,6,0,1,⋯,0,0,0,0,0,0,0,0,0,3


In [78]:
library('dummies')
data_onehot <- dummy.data.frame(datam[,factor_variables], sep="_")

Warning message in dummy.classes == "ALL" || class(data[, nm]) %in% dummy.classes:
"‘length(x) = 2 > 1’ dans la conversion automatique vers ‘logical(1)’"
Warning message in model.matrix.default(~x - 1, model.frame(~x - 1), contrasts = FALSE):
"un argument de contrastes qui n’est pas une liste est ignoré"
Warning message in dummy.classes == "ALL" || class(data[, nm]) %in% dummy.classes:
"‘length(x) = 2 > 1’ dans la conversion automatique vers ‘logical(1)’"
Warning message in model.matrix.default(~x - 1, model.frame(~x - 1), contrasts = FALSE):
"un argument de contrastes qui n’est pas une liste est ignoré"
Warning message in dummy.classes == "ALL" || class(data[, nm]) %in% dummy.classes:
"‘length(x) = 2 > 1’ dans la conversion automatique vers ‘logical(1)’"
Warning message in model.matrix.default(~x - 1, model.frame(~x - 1), contrasts = FALSE):
"un argument de contrastes qui n’est pas une liste est ignoré"
Warning message in dummy.classes == "ALL" || class(data[, nm]) %in% dummy.classes:

In [87]:
head(data_onehot,3)
head(datam_unfactor,3)

,land_surface_condition_n,land_surface_condition_o,land_surface_condition_t,foundation_type_h,foundation_type_i,foundation_type_r,foundation_type_u,foundation_type_w,roof_type_n,roof_type_q,⋯,plan_configuration_m,plan_configuration_n,plan_configuration_o,plan_configuration_q,plan_configuration_s,plan_configuration_u,legal_ownership_status_a,legal_ownership_status_r,legal_ownership_status_v,legal_ownership_status_w
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,0,0,1,0,0,1,0,0,1,0,⋯,0,0,0,0,0,0,0,0,1,0
2,0,0,1,0,0,1,0,0,1,0,⋯,0,0,0,0,0,0,0,0,1,0
3,0,1,0,0,0,1,0,0,0,1,⋯,0,0,0,0,0,0,0,0,1,0


,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,⋯,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,4,30,266,1224,1,25,5,2,0,1,⋯,0,0,0,0,0,0,0,0,0,2
2,8,17,409,12182,2,0,13,7,0,1,⋯,0,0,0,0,0,0,0,0,0,3
3,12,17,716,7056,2,5,12,6,0,1,⋯,0,0,0,0,0,0,0,0,0,3


In [85]:
datam<-cbind(datam_unfactor,data_onehot)
head(datam,3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,⋯,plan_configuration_m,plan_configuration_n,plan_configuration_o,plan_configuration_q,plan_configuration_s,plan_configuration_u,legal_ownership_status_a,legal_ownership_status_r,legal_ownership_status_v,legal_ownership_status_w
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,4,30,266,1224,1,25,5,2,0,1,⋯,0,0,0,0,0,0,0,0,1,0
2,8,17,409,12182,2,0,13,7,0,1,⋯,0,0,0,0,0,0,0,0,1,0
3,12,17,716,7056,2,5,12,6,0,1,⋯,0,0,0,0,0,0,0,0,1,0


Some features are also categorical, but hold the type integer (geo_level_n_id and damage_grade). We will also one-hot encode them.

In [88]:
#colnames(datam)
#colnames(datam) %in% c('geo_level_1_id','geo_level_2_id','geo_level_3_id','damage_grade')
#num_cat_variables<-which(colnames(datam) %in% c('geo_level_1_id','geo_level_2_id','geo_level_3_id','damage_grade'))
#num_cat_variables

Should we normalize the dataset? We lose some interpretation of the final results, but some features have quite different ranges, so it may be necessary for some methods (DNN). We will not normalize binary and categorical features.

In [89]:
library(dplyr)

Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"

Attachement du package : 'dplyr'


Les objets suivants sont masqués depuis 'package:stats':

    filter, lag


Les objets suivants sont masqués depuis 'package:base':

    intersect, setdiff, setequal, union


